# Session 5 — Monitoring Model Explainability and Data Drift using Evidently AI

**Goal:** answer the question every deployed model eventually faces — "is the data
I'm seeing in production still similar to what I trained on?" — using
[Evidently AI](https://www.evidentlyai.com/), an open-source monitoring library.

## Why this matters

A model's accuracy on a held-out test set (Sessions 1, 4) tells you nothing about how
it performs six months later, once real-world data has shifted (new customer
segments, a pandemic, a UI redesign that changes what users click). **Data drift** is
when the distribution of incoming features changes; Evidently computes this
automatically, per column, and renders an interactive HTML report.

## Prerequisites

```bash
pip install evidently
```
Runs entirely locally — no account or cloud service needed. This notebook uses
Evidently's current `Report`/`presets` API (evidently >= 0.4).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from evidently import Report
from evidently.presets import DataDriftPreset

print("Evidently is installed and ready.")

## Step 1 — Simulate a "reference" and a "current" dataset

In a real deployment, **reference** is the training/validation data (the world your
model was built for) and **current** is a recent slice of production traffic. Here we
simulate drift by deliberately shifting two features in the "current" set.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "target"]

rng = np.random.default_rng(42)
reference = df.sample(n=100, random_state=1).reset_index(drop=True)

current = df.drop(reference.index, errors="ignore").sample(n=50, random_state=2).reset_index(drop=True)
# Simulate drift: sensor recalibration shifted sepal_length up, and a data pipeline bug
# started truncating petal_width toward zero.
current["sepal_length"] = current["sepal_length"] + rng.normal(1.5, 0.3, size=len(current))
current["petal_width"] = current["petal_width"] * 0.4

print("reference:", reference.shape, " current:", current.shape)

## Step 2 — Run a Data Drift report

`Report([DataDriftPreset()])` builds the report definition; `.run(current_data=...,
reference_data=...)` executes it and returns a `Snapshot`. Under the hood,
`DataDriftPreset` compares each column's distribution between `reference` and
`current` using a statistical test appropriate to the column type (Kolmogorov-Smirnov
for continuous features here), and rolls the per-column results up into a
`DriftedColumnsCount` summary metric.

In [ ]:
drift_report = Report([DataDriftPreset()])
snapshot = drift_report.run(current_data=current, reference_data=reference)

result = snapshot.dict()
print(f"{len(result['metrics'])} metrics computed:")
for m in result["metrics"]:
    print(" -", m["metric_name"])

## Step 3 — Extract the drift summary and per-column results

The first metric in a `DataDriftPreset` run is always `DriftedColumnsCount`, which
reports how many/what share of columns drifted; every metric after that is a
per-column `ValueDrift` (a p-value from the statistical test — below the configured
`threshold`, typically 0.05, means that column drifted).

In [ ]:
def summarize_drift(snapshot_result):
    drift_count = snapshot_result["metrics"][0]
    n_drifted = drift_count["value"]["count"]
    drift_share = drift_count["value"]["share"]

    column_results = {}
    for m in snapshot_result["metrics"][1:]:
        if m["config"]["type"] == "evidently:metric_v2:ValueDrift":
            column = m["config"]["column"]
            threshold = m["config"]["threshold"]
            p_value = m["value"]
            column_results[column] = {
                "drifted": p_value < threshold,
                "p_value": p_value,
                "method": m["config"]["method"],
            }
    return n_drifted, drift_share, column_results

n_drifted, drift_share, column_results = summarize_drift(result)
print(f"Drifted columns: {int(n_drifted)} ({drift_share:.0%} of all columns)")
print(f"Dataset-level drift flag (share >= 0.5): {drift_share >= 0.5}")
print()
for col, info in column_results.items():
    flag = "DRIFTED" if info["drifted"] else "ok"
    print(f"  {col:<15} {flag:<8} p-value={info['p_value']:.4g}  (method: {info['method']})")

## Step 4 — Save the interactive HTML report

The snapshot also renders a full interactive dashboard — distribution plots per
column, side by side — useful for a human reviewing *why* a column drifted, not just
that it did.

In [ ]:
snapshot.save_html("data_drift_report.html")
print("Saved data_drift_report.html -- open it in a browser to explore interactively.")

## Step 5 — The target column is just another column

Notice `target` already appeared in Step 3's output above (using a chi-square test,
since it's categorical) — `DataDriftPreset` treats the label column the same as any
feature when it's included in the DataFrame, so a separate "target drift" step isn't
needed: whether the label distribution shifted is already part of the same report.

## Step 6 — Model explainability: which features drive predictions?

A lightweight, model-based complement to drift monitoring — which features the model
actually relies on. For a much deeper, per-prediction explanation, see Session 22
(SHAP).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns="target")
y = df["target"]
clf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X, y)

importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importances (a lightweight explainability signal):")
print(importances.round(4))

## What to try next

* Wire `Report.run()` into a scheduled job (cron, Airflow, or the CI/CD pipeline from
  Session 10) that runs nightly against the latest production data slice and alerts
  when `drift_share` crosses your threshold.
* Session 17 builds directly on this: automatically triggering a retraining job the
  moment drift crosses a threshold, instead of just reporting it.
* Combine with Deepchecks (Session 11) for validation checks Evidently doesn't cover
  (schema violations, duplicate rows, label leakage).